In [45]:
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline

In [46]:
df = pd.read_csv('../data/ppr-group-25200353-train-geocoded.csv')
df.head()

,date_of_sale,address,county,price,not_full_market_price,vat_exclusive,property_description,mapbox_query,latitude,longitude,confidence,feature_type,match_code,geometry,index_right,cso_lea
0,2016-02-29,"APT 14, RUSSELL COURT, FELTRIM RD",Dublin,118487.0,No,No,Second-Hand Dwelling house /Apartment,"RUSSELL COURT, FELTRIM RD, CO. DUBLIN",53.433318,-6.177927,none,street,{},POINT (-6.177927 53.433318),86.0,HOWTH-MALAHIDE
1,2016-07-08,"19 PARK VILLAS, DEMESNE RD, DUNDALK",Louth,222000.0,No,No,Second-Hand Dwelling house /Apartment,"19 PARK VILLAS, DEMESNE RD, DUNDALK, CO. LOUTH",54.003768,-6.406065,high,address,"{""address_number"": ""matched"", ""street"": ""match...",POINT (-6.406065 54.003768),66.0,DUNDALK-CARLINGFORD
2,2016-04-15,"No 2 The Grove, Cahereen Heights, Castleisland",Kerry,123348.0,No,Yes,New Dwelling house /Apartment,"THE GROVE, CAHEREEN HEIGHTS, CASTLEISLAND, CO....",52.221088,-9.468242,none,street,{},POINT (-9.468242 52.221088),43.0,CASTLEISLAND
3,2016-12-21,"APT 4 BLOCK 6, WOODFORD, WHEATON HALL",Louth,115000.0,No,No,Second-Hand Dwelling house /Apartment,"BLOCK 6, WOODFORD, WHEATON HALL, CO. LOUTH",53.050891,-8.399598,low,address,"{""address_number"": ""matched"", ""street"": ""unmat...",POINT (-8.399598 53.050891),109.0,LOUGHREA
4,2016-10-11,"18 GERALDINE ST, DUBLIN 7, DUBLIN",Dublin,225500.0,No,No,Second-Hand Dwelling house /Apartment,"18 GERALDINE ST, DUBLIN 7, DUBLIN, CO. DUBLIN",53.357662,-6.270328,high,address,"{""address_number"": ""matched"", ""street"": ""match...",POINT (-6.270328 53.357662),30.0,CABRA-GLASNEVIN


# Only keeping relevant columns

In [47]:
feature_cols = []

More will be added as we engineer features.

## Handling Binary Features

In [48]:
df['is_not_full_market_price'] = df.not_full_market_price.map({'No':0, 'Yes':1})

df['is_vat_exclusive'] = df.vat_exclusive.map({'No':0, 'Yes':1})

df['is_new'] = df['property_description'].map({
    'Second-Hand Dwelling house /Apartment': 0, 
    'New Dwelling house /Apartment': 1
})

df.head()

,date_of_sale,address,county,price,not_full_market_price,vat_exclusive,property_description,mapbox_query,latitude,longitude,confidence,feature_type,match_code,geometry,index_right,cso_lea,is_not_full_market_price,is_vat_exclusive,is_new
0,2016-02-29,"APT 14, RUSSELL COURT, FELTRIM RD",Dublin,118487.0,No,No,Second-Hand Dwelling house /Apartment,"RUSSELL COURT, FELTRIM RD, CO. DUBLIN",53.433318,-6.177927,none,street,{},POINT (-6.177927 53.433318),86.0,HOWTH-MALAHIDE,0,0,0
1,2016-07-08,"19 PARK VILLAS, DEMESNE RD, DUNDALK",Louth,222000.0,No,No,Second-Hand Dwelling house /Apartment,"19 PARK VILLAS, DEMESNE RD, DUNDALK, CO. LOUTH",54.003768,-6.406065,high,address,"{""address_number"": ""matched"", ""street"": ""match...",POINT (-6.406065 54.003768),66.0,DUNDALK-CARLINGFORD,0,0,0
2,2016-04-15,"No 2 The Grove, Cahereen Heights, Castleisland",Kerry,123348.0,No,Yes,New Dwelling house /Apartment,"THE GROVE, CAHEREEN HEIGHTS, CASTLEISLAND, CO....",52.221088,-9.468242,none,street,{},POINT (-9.468242 52.221088),43.0,CASTLEISLAND,0,1,1
3,2016-12-21,"APT 4 BLOCK 6, WOODFORD, WHEATON HALL",Louth,115000.0,No,No,Second-Hand Dwelling house /Apartment,"BLOCK 6, WOODFORD, WHEATON HALL, CO. LOUTH",53.050891,-8.399598,low,address,"{""address_number"": ""matched"", ""street"": ""unmat...",POINT (-8.399598 53.050891),109.0,LOUGHREA,0,0,0
4,2016-10-11,"18 GERALDINE ST, DUBLIN 7, DUBLIN",Dublin,225500.0,No,No,Second-Hand Dwelling house /Apartment,"18 GERALDINE ST, DUBLIN 7, DUBLIN, CO. DUBLIN",53.357662,-6.270328,high,address,"{""address_number"": ""matched"", ""street"": ""match...",POINT (-6.270328 53.357662),30.0,CABRA-GLASNEVIN,0,0,0


In [49]:
feature_cols += ['is_not_full_market_price', 'is_vat_exclusive', 'is_new']

## Handling date_of_sale

### Weekday

In [50]:
df['date_of_sale'] = pd.to_datetime(df['date_of_sale'])

df['day_of_week'] = df['date_of_sale'].dt.day_name().str.lower()

weekday_dummies = pd.get_dummies(
    df.day_of_week, 
    prefix='weekday',
    drop_first=True,
    dtype=int
)

# concatenate the dummy variable columns onto the original DataFrame (axis=0 means rows, axis=1 means columns)
df = pd.concat([df, weekday_dummies], axis=1)

### Month

In [51]:
df['sale_month'] = df['date_of_sale'].dt.month_name().str.lower()

month_dummies = pd.get_dummies(
    df['sale_month'], 
    prefix='month', 
    drop_first=True, 
    dtype=int        
)

df = pd.concat([df, month_dummies], axis=1)

### Year

In [52]:
df['sale_year'] = df['date_of_sale'].dt.year

In [53]:
feature_cols += weekday_dummies.columns.tolist()
feature_cols += month_dummies.columns.tolist()
feature_cols.append('sale_year')
display(feature_cols)

['is_not_full_market_price',
 'is_vat_exclusive',
 'is_new',
 'weekday_monday',
 'weekday_saturday',
 'weekday_sunday',
 'weekday_thursday',
 'weekday_tuesday',
 'weekday_wednesday',
 'month_august',
 'month_december',
 'month_february',
 'month_january',
 'month_july',
 'month_june',
 'month_march',
 'month_may',
 'month_november',
 'month_october',
 'month_september',
 'sale_year']

## Handling local electoral areas

In [54]:
lea_dummies = pd.get_dummies(
    df.cso_lea.str.lower(), 
    prefix='lea',
    drop_first=True,
    dtype=int
)

display(lea_dummies)

# concatenate the dummy variable columns onto the original DataFrame (axis=0 means rows, axis=1 means columns)
df = pd.concat([df, lea_dummies], axis=1)

,lea_ardee,lea_arklow,lea_artane-whitehall,lea_ashbourne,lea_athenry-oranmore,lea_athlone,lea_athy,lea_bailieborough-cootehill,lea_balbriggan,lea_ballina,...,lea_tramore-waterford city west,lea_trim,lea_tuam,lea_tullamore,lea_tullow,lea_waterford city east,lea_waterford city south,lea_westport,lea_wexford,lea_wicklow
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
55435,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
55436,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
55437,0,0,0,0,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0
55438,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [55]:
feature_cols += lea_dummies.columns.tolist()
display(feature_cols[-15:])

['lea_swords',
 'lea_tallaght central',
 'lea_tallaght south',
 'lea_thurles',
 'lea_tralee',
 'lea_tramore-waterford city west',
 'lea_trim',
 'lea_tuam',
 'lea_tullamore',
 'lea_tullow',
 'lea_waterford city east',
 'lea_waterford city south',
 'lea_westport',
 'lea_wexford',
 'lea_wicklow']

# Linear Regression

In [56]:
X = df[feature_cols]
y = df.price

# follow the usual sklearn pattern: import, instantiate, fit
from sklearn.linear_model import LinearRegression
lm = LinearRegression()
lm.fit(X, y)

# print intercept and coefficients
print(lm.intercept_)
print(lm.coef_)

-34905658.73160785
[-9.84969022e+04  1.22269695e+04  1.91061597e+04 -7.66227210e+03
 -2.34694423e+03 -2.29826998e+04 -5.69122919e+03 -1.03746868e+04
 -1.04781673e+04  1.38449366e+04  1.56076008e+04 -4.74988267e+03
 -1.63655352e+02  1.58904763e+04  7.65085937e+03 -3.51472053e+03
 -3.63820125e+03  1.41903057e+04  1.00388617e+04  1.48035499e+04
  1.73800798e+04  1.85052510e+04  5.74757615e+04  1.74799862e+05
  1.00563892e+05  7.22188329e+04 -1.22489117e+04  2.30932974e+04
 -4.04697975e+04  1.59439218e+05 -6.09679724e+04 -5.87977121e+04
 -3.93060034e+04 -3.83410542e+04  1.04875345e+05 -2.61081106e+04
 -6.94202450e+04 -7.08966486e+04  1.13418860e+05  6.51203753e+04
  1.11551695e+05  4.74360391e+04 -6.45923724e+04 -3.16410551e+04
  4.83293493e+05  8.96905121e+04 -3.35804923e+04 -7.65593510e+04
  1.86892322e+05  2.76867337e+05 -5.68892895e+04  2.18360945e+05
 -4.41065709e+04  2.15332821e+04 -1.16957182e+04  1.24109863e+04
 -5.88898174e+04 -6.48112655e+04 -4.94543378e+04  2.16334091e+03
  8.41

In [59]:
lm.score(X, y)

0.3970984806196952

In [60]:
list(zip(feature_cols, lm.coef_))

[('is_not_full_market_price', np.float64(-98496.90223400501)),
 ('is_vat_exclusive', np.float64(12226.969472667668)),
 ('is_new', np.float64(19106.15966334904)),
 ('weekday_monday', np.float64(-7662.2720971273375)),
 ('weekday_saturday', np.float64(-2346.9442286483245)),
 ('weekday_sunday', np.float64(-22982.6998234552)),
 ('weekday_thursday', np.float64(-5691.229194559382)),
 ('weekday_tuesday', np.float64(-10374.68678137286)),
 ('weekday_wednesday', np.float64(-10478.167297281718)),
 ('month_august', np.float64(13844.93659119853)),
 ('month_december', np.float64(15607.600824485067)),
 ('month_february', np.float64(-4749.88266923331)),
 ('month_january', np.float64(-163.65535172152886)),
 ('month_july', np.float64(15890.476255058902)),
 ('month_june', np.float64(7650.859371057824)),
 ('month_march', np.float64(-3514.7205287973484)),
 ('month_may', np.float64(-3638.20125050511)),
 ('month_november', np.float64(14190.305740011725)),
 ('month_october', np.float64(10038.861676653141)),
 (

In [64]:
lower = y.quantile(0)
upper = y.quantile(1)

print(f"Clamped lower: €{lower:,.0f}")
print(f"Clamped upper: €{upper:,.0f}")
print(f"At lower bound: {(y == lower).sum()}")
print(f"At upper bound: {(y == upper).sum()}")

Clamped lower: €5,260
Clamped upper: €1,426,515
At lower bound: 1
At upper bound: 551
